# Kinetic Plasma Dynamics and the Vlasov Equation
### Physics II — BSDSBA 2028 | Theory Notebook

---

> **Abstract.** This notebook develops the kinetic theory of a neutral, collisionless plasma from microscopic first principles, culminating in the Vlasov–Poisson system and its linearised consequences. Beginning from the exact Klimontovich description of $N$ point particles, we perform a controlled mean-field reduction to obtain the Vlasov equation, characterise the geometric incompressibility of its phase-space flow via Liouville's theorem, and derive the dispersion relation for Langmuir waves — including the mechanism of collisionless Landau damping — as the central physical prediction of the theory.

**References:**
- Nicholson, D. R. *Introduction to Plasma Theory*. Wiley, 1983. Chs. 4–6.
- Goldston, R. J. & Rutherford, P. H. *Introduction to Plasma Physics*. IOP, 1995. Chs. 22–23.
- Landau, L. D. (1946). On the vibrations of the electronic plasma. *J. Phys. USSR* **10**, 25.
- Klimontovich, Yu. L. *The Statistical Theory of Non-Equilibrium Processes in a Plasma*. MIT Press, 1967.
- Stix, T. H. *Waves in Plasmas*. AIP, 1992.

---

## Contents

0. [The Fourth State: Defining the Plasma Medium](#0)
1. [Mapping the Mist: The 6D Phase Space Distribution](#1)
2. [From Klimontovich to Vlasov: First-Principles Reduction](#2)
3. [The Geometry of Flow: Liouville's Theorem](#3)
4. [The Self-Consistent Loop: The Vlasov–Poisson System](#4)
5. [The Plasma Shiver: Linearisation for Waves](#5)
6. [The Climax: Deriving Langmuir Waves and Landau Damping](#6)
7. [Bridging to Code: The Need for Numerical Methods](#7)

In [ ]:
# ── Dependencies ───────────────────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib import cm
from scipy.special import wofz
from scipy.optimize import fsolve
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.dpi': 130,
    'font.family': 'serif',
    'mathtext.fontset': 'cm',
    'axes.labelsize': 12,
    'axes.titlesize': 13,
    'legend.fontsize': 10,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
})

# ── Physical constants (SI) ────────────────────────────────────────────────────
e_charge = 1.602e-19    # Elementary charge [C]
m_e      = 9.109e-31    # Electron mass [kg]
eps_0    = 8.854e-12    # Permittivity of free space [F/m]
k_B      = 1.381e-23    # Boltzmann constant [J/K]

# ── Helper functions ───────────────────────────────────────────────────────────
def maxwellian_1d(v, vth=1.0, vd=0.0):
    """Normalised 1-D Maxwellian distribution.

    Parameters
    ----------
    v   : array-like, velocity grid
    vth : float, thermal velocity (standard deviation)
    vd  : float, drift velocity

    Returns
    -------
    f(v) = exp(-(v-vd)^2 / 2vth^2) / (sqrt(2π) vth)
    """
    return np.exp(-0.5 * ((v - vd) / vth)**2) / (np.sqrt(2 * np.pi) * vth)


def plasma_Z(zeta):
    """Plasma dispersion function Z(ζ) = i√π w(ζ), via Faddeeva.

    Parameters
    ----------
    zeta : complex, normalised frequency ζ = ω / (√2 k vth)

    Returns
    -------
    Z(ζ) : complex
    """
    return 1j * np.sqrt(np.pi) * wofz(zeta)


print("Environment ready.")

---
<a id='0'></a>
## 0. The Fourth State: Defining the Plasma Medium

### The Physical Narrative

Matter exists in four fundamental states. The plasma — the fourth and by far the most abundant in the observable universe — is a gas heated to sufficient temperature that electrons are no longer bound to their parent nuclei. The result is a soup of free-roaming positive ions and negative electrons, electrically coupled to one another across macroscopic distances by the long-range Coulomb force.

The key distinction from a neutral gas is interaction topology. In an ordinary gas, a molecule interacts only when it physically collides with a neighbour; between collisions, it is effectively isolated. In a plasma, every charged particle simultaneously attracts or repels every other charged particle across the entire system. Particle dynamics are therefore inherently *collective* rather than binary.

> **Analogy.** If a normal gas is a crowded room of blindfolded people — interacting only when they bump into one another — a plasma is the same room where every person carries a magnet. No one needs to touch; every participant constantly pulls and pushes on every other across the full extent of the space.

### Why a *Neutral* Plasma is Required

The present study is specifically concerned with a **neutral plasma**: one in which the total positive charge of the ion background precisely cancels the total negative charge of the electron population on macroscopic scales. This condition is called **quasineutrality** and is not a simplifying assumption but a physical necessity for any stable plasma in equilibrium.

In the simulation context, the ions are treated as a **fixed, uniform neutralising background** of number density $n_0$. This approximation is justified because the ion mass $m_i \gg m_e$: ions are orders of magnitude more inertial than electrons and respond on a timescale $\omega_{pi}^{-1} \sim \sqrt{m_i/m_e}\,\omega_{pe}^{-1}$ far longer than the electron plasma oscillation period. On the timescales of Langmuir wave dynamics, the ions are essentially frozen. The net charge density that enters Poisson's equation is then

$$\rho = e\!\left(n_0 - \int f_e\, d^3v\right), \tag{0.1}$$

where $f_e$ is the electron distribution function. A non-neutral plasma would carry a net space charge that drives a mean electric field, masking the perturbative oscillatory fields of Langmuir waves and preventing a clean demonstration of the wave physics. **The neutrality condition is therefore the prerequisite for the entire subsequent analysis.**

### The Plasma Parameter: Entrance Condition for Vlasov Theory

The central dimensionless quantity governing the validity of the kinetic description is the **plasma parameter**:

$$\boxed{\Lambda = n\lambda_D^3 \gg 1.} \tag{0.2}$$

**Variable anatomy:**
- $\Lambda$ — the plasma parameter (dimensionless). Represents the number of particles contained within a Debye sphere.
- $n$ — number density of electrons $[\text{m}^{-3}]$. Characterises how densely the plasma is populated.
- $\lambda_D$ — the **Debye length** $[\text{m}]$, defined as

$$\lambda_D = \sqrt{\frac{\epsilon_0 T_e}{n e^2}}, \tag{0.3}$$

where $T_e$ is the electron temperature in energy units and $e$ is the elementary charge. The Debye length is the characteristic distance over which a test charge's electrostatic influence is screened by the surrounding plasma — the range beyond which charges are effectively neutral to one another.

**Provenance.** Equation (0.2) emerges from the requirement for quasineutrality to be self-consistent: collective electromagnetic interactions (proportional to $n\lambda_D^3$) must vastly outnumber binary close encounters (proportional to unity). Only when $\Lambda \gg 1$ can individual particle–particle collisions be neglected in favour of the smooth mean field generated by the entire population.

**Intuition.** The plasma parameter is the admission criterion for Vlasov theory: it asserts that a representative particle experiences the simultaneous, statistically averaged influence of $\Lambda \gg 1$ neighbours rather than a sequence of discrete billiard-ball impacts. This is the physical justification for replacing the exact discrete dynamics with a smooth distribution function.

In [ ]:
# ── Figure 0: Plasma parameter Λ across parameter space ───────────────────────
fig, ax = plt.subplots(figsize=(9, 6))

n_arr = np.logspace(6, 34, 400)   # m^-3
T_arr = np.logspace(-2, 8, 400)   # eV (T in energy units)
N_g, T_g = np.meshgrid(n_arr, T_arr)

# Debye length and plasma parameter
lD  = np.sqrt(eps_0 * T_g * e_charge / (N_g * e_charge**2))
Lam = N_g * lD**3

cf = ax.contourf(N_g, T_g, np.log10(Lam),
                 levels=np.linspace(-2, 12, 60), cmap='RdYlGn', extend='both')
plt.colorbar(cf, ax=ax, label=r'$\log_{10}(\Lambda = n\lambda_D^3)$')
# Vlasov validity boundary: Λ = 1
ax.contour(N_g, T_g, np.log10(Lam), levels=[0], colors='k', linewidths=2.5)
ax.text(3e8, 5e-2,
        r'$\Lambda < 1$: Collisions dominate' + '\n' + r'Vlasov theory invalid',
        fontsize=9, color='darkred',
        bbox=dict(boxstyle='round', fc='lightyellow', alpha=0.95))

# Annotate representative plasmas
plasmas = {
    'Solar wind':   (5e6,   10),
    'Tokamak':      (1e20,  1e4),
    'ICF':          (1e31,  1e7),
    'Ionosphere':   (1e11,  0.1),
    'Laser plasma': (1e27,  1e3),
}
for name, (n, T) in plasmas.items():
    ax.plot(n, T, 'ko', ms=7, zorder=5)
    ax.annotate(name, (n, T), xytext=(6, 5), textcoords='offset points', fontsize=8)

ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlabel(r'Number density $n\;[\mathrm{m}^{-3}]$')
ax.set_ylabel(r'Temperature $T_e\;[\mathrm{eV}]$')
ax.set_title(
    r'Figure 0 — Plasma Parameter $\Lambda = n\lambda_D^3$: '
    r'Vlasov theory is valid in the green region ($\Lambda \gg 1$)',
    fontweight='bold')
ax.grid(alpha=0.2, which='both')
plt.tight_layout()
plt.show()

---
<a id='1'></a>
## 1. Mapping the Mist: The 6D Phase Space Distribution

### The Physical Narrative

A plasma of $N \sim 10^{20}$ particles cannot be described particle-by-particle; the state space is of dimension $6N$, utterly beyond any analytical or computational approach. Instead, we adopt a statistical description: rather than asking where each individual electron *is*, we ask how many electrons are *here* and *moving like this* at each instant.

> **Analogy.** Consider a busy international airport terminal. Rather than following the itinerary of every individual passenger, airport operations planners maintain a *density map*: a function that records how many people are in each corridor (position $\mathbf{x}$) moving toward their gate at each walking speed (velocity $\mathbf{v}$) at each moment (time $t$). The density map suffices for all macroscopic decisions — crowd control, flow optimisation — without requiring knowledge of any individual's identity.

The plasma distribution function is precisely this density map, promoted to a six-dimensional space.

### The Distribution Function

The **distribution function** $f(\mathbf{r}, \mathbf{v}, t)$ is defined such that

$$\boxed{dN = f(\mathbf{r}, \mathbf{v}, t)\, d^3\mathbf{r}\, d^3\mathbf{v}} \tag{1.1}$$

gives the number of particles occupying the infinitesimal phase-space volume $d^3\mathbf{r}\,d^3\mathbf{v}$ centred at $(\mathbf{r}, \mathbf{v})$ at time $t$.

**Variable anatomy:**
- $f(\mathbf{r}, \mathbf{v}, t)$ — the distribution function $[\mathrm{m}^{-6}\mathrm{s}^3]$. Its value at a point in phase space is the local particle number density in that 6D volume.
- $\mathbf{r} \in \mathbb{R}^3$ — position vector $[\mathrm{m}]$. Three spatial coordinates specifying *where* the particle is.
- $\mathbf{v} \in \mathbb{R}^3$ — velocity vector $[\mathrm{m\,s}^{-1}]$. Three velocity components specifying *how fast and in which direction* the particle moves.
- $t$ — time $[\mathrm{s}]$.
- $d^3\mathbf{r}\,d^3\mathbf{v}$ — the six-dimensional phase-space volume element $[\mathrm{m}^3 \cdot \mathrm{m}^3\mathrm{s}^{-3}]$.

**Provenance.** The definition (1.1) is the continuum limit of the discrete particle description: as $N \to \infty$ with the product $n = N/V$ held fixed, the sharp point masses blur into a smooth density cloud over phase space.

**Conceptual subtlety.** In equation (1.1), $\mathbf{r}$, $\mathbf{v}$, and $t$ are **independent variables** — labels on the phase-space map. This is a fundamental departure from Newtonian mechanics, where the particle trajectory $\mathbf{r}(t)$, $\mathbf{v}(t)$ treats position and velocity as *dependent* functions of time. The kinetic description does not follow individual particles; it describes the density of the entire population simultaneously.

### Macroscopic Moments

All experimentally accessible fluid quantities are recovered as velocity-space **moments** of $f$:

| Moment | Expression | Observable |
|--------|-----------|------------|
| Zeroth | $n(\mathbf{r},t) = \displaystyle\int f\,d^3v$ | Number density $[\mathrm{m}^{-3}]$ |
| First  | $n\mathbf{u} = \displaystyle\int \mathbf{v}\,f\,d^3v$ | Bulk velocity $[\mathrm{m\,s}^{-1}]$ |
| Second | $\overleftrightarrow{P} = m\displaystyle\int(\mathbf{v}-\mathbf{u})(\mathbf{v}-\mathbf{u})f\,d^3v$ | Pressure tensor $[\mathrm{Pa}]$ |
| Energy | $\mathcal{E} = \frac{m}{2}\displaystyle\int v^2 f\,d^3v$ | Kinetic energy density $[\mathrm{J\,m}^{-3}]$ |

The kinetic description is strictly *richer* than any fluid model: fluid equations correspond to a finite truncation of the moment hierarchy, discarding information about the shape of $f$ in velocity space. Non-Maxwellian features — beams, loss cones, resonant velocity-space structures — are invisible to fluid theory but fully resolved by the distribution function.

In [ ]:
# ── Figure 1: The distribution function and its moments ───────────────────────
v = np.linspace(-5, 5, 600)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# --- Panel 1: Maxwellian and its moments labelled ---
ax = axes[0]
fM = maxwellian_1d(v, vth=1.0)
ax.plot(v, fM, lw=2.5, color='steelblue', label=r'$f_0(v)$: Maxwellian')
# Shade to indicate the zeroth moment (area = n)
ax.fill_between(v, fM, alpha=0.15, color='steelblue', label=r'Area $= n$ (zeroth moment)')
# Mark mean (first moment)
ax.axvline(0, color='firebrick', lw=1.5, ls='--', label=r'$\langle v\rangle = 0$ (first moment)')
# Mark thermal width (second moment)
ax.axvspan(-1, 1, alpha=0.1, color='seagreen', label=r'$\pm v_{th}$ (second moment)')
ax.set_xlabel(r'$v / v_{th}$'); ax.set_ylabel(r'$f(v)$')
ax.set_title('Distribution Function and Moments')
ax.legend(fontsize=8); ax.grid(alpha=0.3)

# --- Panel 2: Non-Maxwellian — beam + core ---
ax = axes[1]
f_core = maxwellian_1d(v, vth=1.0)
f_beam = 0.12 * maxwellian_1d(v, vth=0.35, vd=3.2)
f_tot  = f_core + f_beam
dfdv   = np.gradient(f_tot, v)
ax.plot(v, f_core, lw=2, color='steelblue', ls='--', label='Core')
ax.plot(v, f_beam, lw=2, color='firebrick',  ls='--', label='Beam')
ax.plot(v, f_tot,  lw=2.5, color='k',        label='Total $f$')
ax.fill_between(v, f_tot, where=dfdv > 0, alpha=0.25, color='gold',
                label=r'$\partial f/\partial v > 0$: free energy')
ax.set_xlabel(r'$v / v_{th}$'); ax.set_ylabel(r'$f(v)$')
ax.set_title('Non-Maxwellian: Beam–Core'); ax.legend(fontsize=8); ax.grid(alpha=0.3)

# --- Panel 3: 2-D phase space snapshot f(x, v) ---
ax = axes[2]
xg = np.linspace(0, 2 * np.pi, 300)
vg = np.linspace(-3.5, 3.5, 300)
X, V = np.meshgrid(xg, vg)
F = maxwellian_1d(V) * (1 + 0.45 * np.cos(X))
c = ax.contourf(X, V, F, levels=25, cmap='plasma')
plt.colorbar(c, ax=ax, label=r'$f(x, v, t_0)$')
ax.set_xlabel(r'$x$ [arb.]'); ax.set_ylabel(r'$v / v_{th}$')
ax.set_title(r'Phase Space Density $f(x, v)$')

plt.suptitle('Figure 1 — The Distribution Function: the complete statistical description of the plasma',
             fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

---
<a id='2'></a>
## 2. From Klimontovich to Vlasov: First-Principles Reduction

### The Physical Narrative

We begin with a description that sees every particle as an infinitely sharp point mass — the Klimontovich picture. To make this tractable, we then perform a controlled "squinting" operation, blurring the discrete spikes into a smooth continuum. The key choice is *what level of detail to retain* and *what to discard*.

> **Analogy.** Consider a digital photograph. Zoomed in to individual pixels, the image is a grid of discrete, sharp dots — the Klimontovich description. Zoomed out, the discrete pixels blur into a smooth, continuous image of a face — the Vlasov description. The zoom level is controlled by the plasma parameter $\Lambda$: when $\Lambda \gg 1$, the "pixels" are so densely packed that the smooth approximation is indistinguishable from the truth.

### 2.1 The Klimontovich Distribution

For a species $\alpha$ of $N_0$ particles, the exact microscopic distribution is

$$N_\alpha(\mathbf{x}, \mathbf{v}, t) = \sum_{k=1}^{N_0} \delta^3\!\left[\mathbf{x} - \mathbf{x}_k(t)\right]\delta^3\!\left[\mathbf{v} - \mathbf{v}_k(t)\right]. \tag{2.1}$$

**Variable anatomy:**
- $N_\alpha$ — Klimontovich distribution function for species $\alpha$ $[\mathrm{m}^{-6}\mathrm{s}^3]$. It is a sum of infinitely sharp spikes, one per particle.
- $\delta^3[\cdot]$ — three-dimensional Dirac delta function $[\mathrm{m}^{-3}]$ or $[\mathrm{m}^{-3}\mathrm{s}^{3}]$. Represents a spike of unit weight at the exact location of particle $k$ in position or velocity space.
- $\mathbf{x}_k(t)$, $\mathbf{v}_k(t)$ — position and velocity of the $k$-th particle at time $t$, solutions to Newton's equations.

**Provenance.** Taking the total time derivative of (2.1) and using Newton's law $\dot{\mathbf{v}}_k = \mathbf{F}_k/m$ yields the **Klimontovich equation**, which is exact:

$$\frac{\partial N_\alpha}{\partial t} + \mathbf{v}\cdot\frac{\partial N_\alpha}{\partial \mathbf{x}} + \frac{\mathbf{F}}{m}\cdot\frac{\partial N_\alpha}{\partial \mathbf{v}} = 0. \tag{2.2}$$

This is exact but contains the full microscopic detail — including all binary collisions — encoded in the singular fields $\mathbf{F}$.

### 2.2 The Mean-Field Limit

We decompose every fluctuating quantity into a **smooth ensemble average** and a **fluctuating part**:

$$N_\alpha = \langle N_\alpha \rangle + \delta N_\alpha, \qquad \mathbf{F} = \langle \mathbf{F} \rangle + \delta\mathbf{F}. \tag{2.3}$$

Taking the ensemble average of (2.2) generates a term $\langle \delta\mathbf{F}\cdot\partial\delta N_\alpha/\partial\mathbf{v}\rangle$ representing the correlation between field fluctuations and density fluctuations — the **collision term**. In the mean-field limit $\Lambda \gg 1$, this correlation is of order $\Lambda^{-1}$ relative to the smooth terms and is **neglected**. Setting $f \equiv \langle N_\alpha \rangle$ and $\mathbf{E} \equiv \langle \delta\mathbf{F}/q\rangle$, one obtains the Vlasov equation directly.

The physical content of this step is that each particle interacts with the *average* field of all others, not with their individual positions. This is the mean-field approximation, and it is exact in the limit $\Lambda \to \infty$.

In [ ]:
# ── Figure 2: Klimontovich → Vlasov: microscopic vs smooth ────────────────────
rng = np.random.default_rng(42)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

v_grid = np.linspace(-4, 4, 300)
f_true = maxwellian_1d(v_grid, vth=1.0)

for i, (N_particles, title) in enumerate([
        (30,    r'Klimontovich: $N=30$ (discrete)'),
        (300,   r'Intermediate: $N=300$'),
        (3000,  r'Vlasov limit: $N=3000$ (smooth)')]):
    ax = axes[i]
    particles = rng.standard_normal(N_particles)   # draw from Maxwellian
    ax.hist(particles, bins=30, density=True, color='steelblue',
            alpha=0.55, label=f'Particle histogram')
    ax.plot(v_grid, f_true, 'r-', lw=2, label=r'$f_0(v)$ Maxwellian')
    # Add delta-spike markers for small N
    if i == 0:
        ax.vlines(particles, 0, 0.06, color='k', lw=0.8, alpha=0.7,
                  label=r'$\delta$-spikes')
    ax.set_xlabel(r'$v / v_{th}$'); ax.set_ylabel(r'$f(v)$')
    ax.set_title(title); ax.legend(fontsize=8); ax.grid(alpha=0.3)

plt.suptitle(
    'Figure 2 — From Klimontovich (discrete) to Vlasov (smooth): '
    r'convergence as $N \to \infty$ ($\Lambda \gg 1$)',
    fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

---
<a id='3'></a>
## 3. The Geometry of Flow: Liouville's Theorem

### The Physical Narrative

Once the mean-field limit is adopted, the plasma's distribution function behaves as an **incompressible fluid in six-dimensional phase space**. No matter how violently electromagnetic forces stir and stretch the phase-space density, they can never compress it. This geometric fact is Liouville's theorem, and it is the foundation upon which the Vlasov equation rests.

> **Analogy — The Water-Bag Model.** Imagine the distribution function as a bag of water in six-dimensional space. You may squeeze the bag, stretch it into a long ribbon, or wrap it into a spiral — the shape can change arbitrarily — but the total *volume* of water inside is absolutely fixed. Electromagnetic forces are the hands squeezing the bag; they can redistribute the phase-space density but can never create or destroy it, and crucially, can never compress it to a smaller volume.

### 3.1 The Six-Dimensional Continuity Equation

We treat $f(\mathbf{x}, \mathbf{v}, t)$ as the density of a fluid flowing through phase space with a **6D velocity field**

$$\mathbf{w} = (\dot{\mathbf{x}},\, \dot{\mathbf{v}}) = \left(\mathbf{v},\, \frac{\mathbf{F}}{m}\right) \in \mathbb{R}^6. \tag{3.1}$$

**Variable anatomy:**
- $\mathbf{w} = (\dot{\mathbf{x}}, \dot{\mathbf{v}})$ — the phase-space velocity vector $[\mathrm{m\,s}^{-1},\, \mathrm{m\,s}^{-2}]$. Its first three components are the ordinary velocity; the last three are the acceleration from the Lorentz force.
- $\mathbf{F} = q(\mathbf{E} + \mathbf{v}\times\mathbf{B}/c)$ — the Lorentz force $[\mathrm{N}]$.
- $m$ — particle mass $[\mathrm{kg}]$.

Particle number conservation in an arbitrary 6D volume $\mathcal{V}$ with boundary $\partial\mathcal{V}$ requires

$$\frac{\partial}{\partial t}\int_{\mathcal{V}} f\,d^6w = -\oint_{\partial\mathcal{V}} f\,\mathbf{w}\cdot\hat{n}\,dS_5, \tag{3.2}$$

where $dS_5$ is the five-dimensional surface area element on $\partial\mathcal{V}$ and $\hat{n}$ is its outward unit normal. Applying the 6D divergence theorem to the right-hand side and using the fact that $\mathcal{V}$ is arbitrary:

$$\frac{\partial f}{\partial t} + \nabla_{\mathbf{x}}\cdot(\mathbf{v}\,f) + \nabla_{\mathbf{v}}\cdot\!\left(\frac{\mathbf{F}}{m}\,f\right) = 0. \tag{3.3}$$

This is the **6D continuity equation** for the phase-space fluid.

### 3.2 Liouville's Theorem: Incompressibility of the Phase-Space Flow

**Theorem.** Under Hamiltonian (here, Lorentz-force) dynamics, the phase-space velocity field $\mathbf{w}$ is divergence-free:

$$\nabla_{\mathbf{w}}\cdot\dot{\mathbf{w}} \equiv \nabla_{\mathbf{x}}\cdot\mathbf{v} + \nabla_{\mathbf{v}}\cdot\frac{\mathbf{F}}{m} = 0. \tag{3.4}$$

**Proof.**

*Part 1:* $\nabla_{\mathbf{x}}\cdot\mathbf{v} = 0$, since $\mathbf{x}$ and $\mathbf{v}$ are independent coordinates in phase space — velocity components do not depend on position coordinates in the same phase-space point.

*Part 2:* For the Lorentz force $\mathbf{F} = q(\mathbf{E} + \mathbf{v}\times\mathbf{B}/c)$, we compute

$$\nabla_{\mathbf{v}}\cdot\mathbf{F} = q\,\nabla_{\mathbf{v}}\cdot\mathbf{E} + \frac{q}{c}\,\nabla_{\mathbf{v}}\cdot(\mathbf{v}\times\mathbf{B}). \tag{3.5}$$

The first term vanishes because $\mathbf{E}(\mathbf{x},t)$ does not depend on $\mathbf{v}$: $\nabla_{\mathbf{v}}\cdot\mathbf{E} = 0$.

For the second term, we apply the vector identity $\nabla\cdot(\mathbf{A}\times\mathbf{B}) = \mathbf{B}\cdot(\nabla\times\mathbf{A}) - \mathbf{A}\cdot(\nabla\times\mathbf{B})$ with $\mathbf{A} = \mathbf{v}$ and $\mathbf{B} = \mathbf{B}$:

$$\nabla_{\mathbf{v}}\cdot(\mathbf{v}\times\mathbf{B}) = \mathbf{B}\cdot(\nabla_{\mathbf{v}}\times\mathbf{v}) - \mathbf{v}\cdot(\nabla_{\mathbf{v}}\times\mathbf{B}). \tag{3.6}$$

Now, $\nabla_{\mathbf{v}}\times\mathbf{v} = \mathbf{0}$ (the curl of any vector with respect to itself vanishes), and $\nabla_{\mathbf{v}}\times\mathbf{B} = \mathbf{0}$ since $\mathbf{B}(\mathbf{x},t)$ is independent of $\mathbf{v}$. Therefore the entire expression (3.6) is zero, and

$$\nabla_{\mathbf{v}}\cdot\frac{\mathbf{F}}{m} = 0. \quad \square \tag{3.7}$$

### 3.3 The Vlasov Equation

Because $\nabla_{\mathbf{w}}\cdot\dot{\mathbf{w}} = 0$, we can write $\nabla_{\mathbf{v}}\cdot(\mathbf{F}f/m) = (\mathbf{F}/m)\cdot\nabla_{\mathbf{v}}f$ and similarly for the $\mathbf{v}$ term. Substituting into (3.3):

$$\boxed{\frac{\partial f}{\partial t} + \mathbf{v}\cdot\frac{\partial f}{\partial\mathbf{x}} + \frac{q}{m}\!\left(\mathbf{E} + \frac{\mathbf{v}\times\mathbf{B}}{c}\right)\!\cdot\frac{\partial f}{\partial\mathbf{v}} = 0.} \tag{3.8}$$

This is the **Vlasov equation**. It is a first-order linear PDE in the six-dimensional space $(\mathbf{x}, \mathbf{v})$.

### 3.4 Characteristic Form: $df/dt = 0$ on Particle Orbits

The Vlasov equation can be written compactly as

$$\frac{df}{dt}\bigg|_{\text{orbit}} = \frac{\partial f}{\partial t} + \frac{d\mathbf{x}}{dt}\cdot\frac{\partial f}{\partial\mathbf{x}} + \frac{d\mathbf{v}}{dt}\cdot\frac{\partial f}{\partial\mathbf{v}} = 0. \tag{3.9}$$

**Interpretation.** The phase-space density $f$ is constant along each particle's trajectory. A parcel of phase-space fluid may be stretched, folded, and filamented by the electromagnetic forces, but it carries its value of $f$ with it unchanged — exactly as an incompressible fluid carries its density.

### 3.5 Consequences: Entropy Conservation and Time Reversibility

Because $f$ is conserved along characteristics, so is any function $G(f)$. In particular, the **Boltzmann entropy**

$$S_{\rm B} = -\int d^3x\,d^3v\, f\ln f \tag{3.10}$$

is an exact constant of the Vlasov equation. The collisionless plasma therefore **never thermalises on its own**: it retains a complete memory of its initial conditions and its entropy never increases. This is in stark contrast to the collisional Boltzmann equation, for which Boltzmann's H-theorem guarantees monotonic entropy increase toward the Maxwellian.

Time-reversibility is the direct corollary: if $(f(\mathbf{x}, \mathbf{v}, t), \mathbf{E}, \mathbf{B})$ is a solution, so is $(f(\mathbf{x}, -\mathbf{v}, -t), -\mathbf{E}, \mathbf{B})$. The system has no preferred arrow of time in the absence of collisions.

In [ ]:
# ── Figure 3: Liouville's theorem — area-preserving phase-space flow ───────────

def pendulum_rhs(state):
    """RHS for a simple pendulum: dx/dt = v, dv/dt = -sin(x)."""
    return np.array([state[1], -np.sin(state[0])])


def rk4_step(state, dt):
    """Single RK4 integration step."""
    k1 = pendulum_rhs(state)
    k2 = pendulum_rhs(state + 0.5 * dt * k1)
    k3 = pendulum_rhs(state + 0.5 * dt * k2)
    k4 = pendulum_rhs(state + dt * k3)
    return state + (dt / 6.0) * (k1 + 2*k2 + 2*k3 + k4)


# Initialise a circular blob of N_p particles
rng = np.random.default_rng(0)
N_p    = 900
theta  = rng.uniform(0, 2 * np.pi, N_p)
radius = 0.28 * np.sqrt(rng.uniform(0, 1, N_p))
states = np.column_stack([
    1.0 + radius * np.cos(theta),
    0.5 + radius * np.sin(theta)
])

# Evolve and snapshot
dt = 0.05
snap_at = [0, 20, 55, 110]
snaps, t_vals = [], []
for step in range(max(snap_at) + 1):
    if step in snap_at:
        snaps.append(states.copy())
        t_vals.append(step * dt)
    states = np.array([rk4_step(s, dt) for s in states])

fig, axes = plt.subplots(1, 4, figsize=(14, 3.8), sharey=True)
colors = cm.plasma(np.linspace(0.15, 0.85, 4))

for i, (snap, t, col) in enumerate(zip(snaps, t_vals, colors)):
    axes[i].scatter(snap[:, 0], snap[:, 1], s=4, color=col, alpha=0.85)
    axes[i].set_title(f'$t = {t:.2f}$', fontsize=11)
    axes[i].set_xlabel(r'$x$', fontsize=11)
    axes[i].set_xlim(-3.2, 3.8)
    axes[i].set_ylim(-2.8, 2.8)
    axes[i].grid(alpha=0.3)
    # Annotate area preservation
    from scipy.spatial import ConvexHull
    try:
        hull = ConvexHull(snap)
        area = hull.volume   # in 2D, .volume gives area
        axes[i].text(0.05, 0.95, f'Area≈{area:.2f}',
                     transform=axes[i].transAxes, fontsize=8, va='top',
                     bbox=dict(boxstyle='round', fc='white', alpha=0.8))
    except Exception:
        pass

axes[0].set_ylabel(r'$v$', fontsize=11)
plt.suptitle(
    r'Figure 3 — Liouville: phase-space blob deforms but preserves area ($\ddot{x}=-\sin x$). '
    r'Convex-hull area $\approx$ const confirms incompressibility.',
    fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
<a id='4'></a>
## 4. The Self-Consistent Loop: The Vlasov–Poisson System

### The Physical Narrative

The Vlasov equation (3.8) governs how the distribution function $f$ evolves under a given electric field $\mathbf{E}$, but does not specify what that field is. The field must itself be generated by the charge distribution of the very particles described by $f$ — a self-referential, nonlinear feedback loop.

> **Analogy — The Flash Mob.** The dancers collectively create the music (the field) through their movement, and the music simultaneously tells each dancer how to move next. Neither the music nor the choreography is primary; they are co-determined, moment by moment.

### 4.1 The Vlasov–Poisson System

In the electrostatic limit appropriate for Langmuir waves (no magnetic perturbations, $v_\phi \ll c$), the system is closed by coupling the Vlasov equation to Poisson's equation:

$$\frac{\partial f}{\partial t} + \mathbf{v}\cdot\nabla_{\mathbf{x}}f + \frac{q\mathbf{E}}{m}\cdot\nabla_{\mathbf{v}}f = 0, \tag{4.1}$$

$$-\Delta\phi = \frac{e}{\epsilon_0}\!\left(n_0 - \int f_e\,d^3v\right), \qquad \mathbf{E} = -\nabla\phi. \tag{4.2}$$

**Variable anatomy:**
- $\phi(\mathbf{x}, t)$ — electrostatic potential $[\mathrm{V}]$. The scalar field from which $\mathbf{E}$ is derived.
- $\mathbf{E} = -\nabla\phi$ — electric field $[\mathrm{V\,m}^{-1}]$. The force per unit charge on the electrons.
- $n_0$ — uniform ion background density $[\mathrm{m}^{-3}]$. Fixed in the simulation; provides global neutrality.
- $\int f_e\,d^3v = n_e(\mathbf{x},t)$ — local electron number density $[\mathrm{m}^{-3}]$, obtained as the zeroth moment of the distribution.
- $\epsilon_0$ — permittivity of free space $[\mathrm{F\,m}^{-1}]$.

**Provenance.** Equation (4.2) is Gauss's law of electrostatics $\nabla\cdot\mathbf{E} = \rho/\epsilon_0$ with the charge density $\rho = e(n_0 - n_e)$. The neutral plasma condition ensures that on average $n_e = n_0$ and $\rho = 0$, so there is no mean electric field — only the perturbation field driven by local density fluctuations.

**The self-consistent loop:**
1. $f \xrightarrow{\int d^3v} n_e$ — integrate the distribution to obtain the electron density.
2. $n_e \xrightarrow{\text{Poisson}} \phi \xrightarrow{-\nabla} \mathbf{E}$ — solve Poisson's equation for the self-consistent field.
3. $\mathbf{E} \xrightarrow{\text{Vlasov}} f$ — evolve the distribution under that field.
4. Return to step 1 at the next timestep.

This loop is the complete Vlasov–Poisson system. It is nonlinear because $\mathbf{E}$ depends on $f$, which in turn evolves under $\mathbf{E}$.

In [ ]:
# ── Figure 4: Schematic of the self-consistent Vlasov–Poisson loop ─────────────
fig, ax = plt.subplots(figsize=(10, 4))
ax.axis('off')

# Draw boxes
boxes = {
    'f':    (0.10, 0.40, r'$f(\mathbf{x},\mathbf{v},t)$' + '\nDistribution'),
    'n':    (0.38, 0.40, r'$n_e(\mathbf{x},t) = \int f\,d^3v$' + '\nDensity'),
    'phi':  (0.65, 0.40, r'$\phi$ via Poisson' + r'$\;\nabla^2\phi = -\rho/\epsilon_0$'),
    'E':    (0.65, 0.10, r'$\mathbf{E} = -\nabla\phi$' + '\nElectric field'),
    'back': (0.10, 0.10, r'Advance $f$ one step' + '\nVlasov equation'),
}
box_style = dict(boxstyle='round,pad=0.4', fc='#dbeafe', ec='#1d4ed8', lw=1.5)
for key, (x, y, label) in boxes.items():
    ax.text(x + 0.12, y + 0.08, label, transform=ax.transAxes,
            ha='center', va='center', fontsize=9,
            bbox=box_style)

# Arrows
arrow_kw = dict(transform=ax.transAxes, arrowprops=dict(
    arrowstyle='->', color='#1d4ed8', lw=2), annotation_clip=False)
arrows = [
    ((0.25, 0.48), (0.36, 0.48)),   # f → n
    ((0.62, 0.48), (0.63, 0.48)),   # n → phi
    ((0.77, 0.40), (0.77, 0.20)),   # phi → E (downward)
    ((0.63, 0.15), (0.26, 0.15)),   # E → back-Vlasov
    ((0.22, 0.18), (0.22, 0.38)),   # Vlasov → f (upward)
]
for (x0, y0), (x1, y1) in arrows:
    ax.annotate('', xy=(x1, y1), xytext=(x0, y0), **arrow_kw)

ax.text(0.5, 0.97, 'Figure 4 — The Vlasov–Poisson Self-Consistent Loop',
        transform=ax.transAxes, ha='center', va='top',
        fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

---
<a id='5'></a>
## 5. The Plasma Shiver: Linearisation for Waves

### The Physical Narrative

The Vlasov–Poisson system is nonlinear: $\mathbf{E}$ depends on $f$, and $f$ is driven by $\mathbf{E}$. Solving the full nonlinear problem requires numerical simulation. However, for small-amplitude perturbations, we can *linearise* the system — reducing it to a tractable set of equations whose solutions reveal the plasma's natural oscillatory modes.

> **Analogy — Plucking a Guitar String.** The resting string represents the equilibrium $f_0$. When plucked gently, it vibrates in its natural modes — sinusoidal waves at characteristic frequencies. The vibration $f_1$ is small; it obeys a linear equation (the wave equation), and its solutions are precisely those modes. If plucked hard enough that the displacement is no longer small, nonlinear effects appear (harmonics, mode coupling). Linearisation captures the small-displacement physics exactly.

### 5.1 The Perturbation Ansatz

We decompose the distribution function and electrostatic potential into a **steady equilibrium** and a **small perturbation**:

$$f(\mathbf{x}, \mathbf{v}, t) = f_0(\mathbf{v}) + f_1(\mathbf{x}, \mathbf{v}, t), \quad |f_1| \ll f_0. \tag{5.1}$$
$$\phi(\mathbf{x}, t) = 0 + \phi_1(\mathbf{x}, t). \tag{5.2}$$

**Variable anatomy:**
- $f_0(\mathbf{v})$ — the equilibrium distribution $[\mathrm{m}^{-6}\mathrm{s}^3]$. Spatially uniform; depends only on speed, not position. Typically taken to be a Maxwellian.
- $f_1(\mathbf{x}, \mathbf{v}, t)$ — the perturbation $[\mathrm{m}^{-6}\mathrm{s}^3]$. Small: $\|f_1\|/\|f_0\| \sim \epsilon \ll 1$.
- $\phi_1(\mathbf{x}, t)$ — the perturbation potential $[\mathrm{V}]$. Zero in equilibrium because the plasma is exactly neutral: $n_e = n_0$.

**Why $\phi_0 = 0$?** In the neutral equilibrium, the ion background charge density exactly cancels the electron charge density at every point: $e(n_0 - n_0) = 0$. Poisson's equation then gives $\nabla^2\phi_0 = 0$ with the physically relevant solution $\phi_0 = 0$ (no mean field).

### 5.2 Derivation of the Linearised Vlasov Equation

Substituting (5.1)–(5.2) into the Vlasov–Poisson system and retaining only **first-order** terms in $f_1$ and $\phi_1$ (discarding the second-order product $f_1 E_1$):

$$\frac{\partial(f_0 + f_1)}{\partial t} + \mathbf{v}\cdot\nabla_{\mathbf{x}}(f_0 + f_1) - \frac{e}{m}\nabla\phi_1\cdot\nabla_{\mathbf{v}}(f_0 + f_1) = 0. \tag{5.3}$$

Since $f_0$ is time-independent and spatially uniform, $\partial f_0/\partial t = 0$ and $\nabla_{\mathbf{x}}f_0 = 0$. The term $\nabla\phi_1\cdot\nabla_{\mathbf{v}}f_1$ is second-order and is dropped. What remains is the **linearised Vlasov equation**:

$$\boxed{\frac{\partial f_1}{\partial t} + \mathbf{v}\cdot\nabla_{\mathbf{x}}f_1 - \frac{e}{m}\nabla\phi_1\cdot\nabla_{\mathbf{v}}f_0 = 0.} \tag{5.4}$$

The corresponding linearised Poisson equation is

$$\nabla^2\phi_1 = \frac{e}{\epsilon_0}\int f_1\,d^3v. \tag{5.5}$$

**Structural observation.** The linearised system (5.4)–(5.5) is now *linear* in $f_1$ and $\phi_1$. The coupling between them is through the term $\nabla\phi_1\cdot\nabla_{\mathbf{v}}f_0$: the perturbation potential drives velocity-space gradients of the equilibrium distribution — precisely the mechanism by which a wave accelerates or decelerates resonant particles.

### 5.3 Fourier–Laplace Analysis: Normal Modes

Because the linearised system has constant (in space and time) coefficients for the equilibrium $f_0(\mathbf{v})$, we seek **plane-wave normal-mode solutions**. For a 1D geometry:

$$f_1(x, v, t) = \hat{f}_1(v)\,e^{i(kx - \omega t)}, \qquad \phi_1(x, t) = \hat{\phi}\,e^{i(kx - \omega t)}. \tag{5.6}$$

**Variable anatomy:**
- $k$ — wave number $[\mathrm{m}^{-1}]$. Specifies the spatial periodicity: $\lambda = 2\pi/k$.
- $\omega$ — complex angular frequency $[\mathrm{rad\,s}^{-1}]$. Its real part $\omega_r$ gives the oscillation frequency; its imaginary part $\gamma = \mathrm{Im}(\omega)$ gives the growth ($\gamma > 0$) or decay ($\gamma < 0$) rate.
- $\hat{\phi}$ — complex amplitude of the potential mode $[\mathrm{V}]$.

Substituting (5.6) into (5.4):

$$-i\omega\hat{f}_1 + ikv\hat{f}_1 - \frac{e}{m}(ik\hat{\phi})\frac{\partial f_0}{\partial v} = 0
\quad\Longrightarrow\quad
\hat{f}_1(v) = \frac{e\hat{\phi}}{m}\frac{k\,\partial f_0/\partial v}{\omega - kv}. \tag{5.7}$$

Substituting into the Fourier-transformed Poisson equation $-k^2\hat{\phi} = (e/\epsilon_0)\int\hat{f}_1\,dv$ and cancelling $\hat{\phi} \neq 0$:

$$\boxed{\varepsilon(k, \omega) \equiv 1 - \frac{\omega_{pe}^2}{k^2}\int_{-\infty}^{\infty} \frac{\partial f_0/\partial v}{v - \omega/k}\,dv = 0.} \tag{5.8}$$

This is the **electrostatic dispersion relation**: a transcendental equation whose solutions $\omega(k)$ are the natural oscillation frequencies of the plasma.

In [ ]:
# ── Figure 5: Linearisation — equilibrium, perturbation, and resonance ─────────
v = np.linspace(-5, 5, 600)

fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))

# --- Panel 1: Equilibrium f0 and perturbation f1 ---
ax = axes[0]
f0 = maxwellian_1d(v, vth=1.0)
f1 = 0.18 * f0 * np.cos(3 * v)   # schematic spatial-Fourier mode
ax.plot(v, f0,       lw=2.5, color='steelblue', label=r'$f_0(v)$: equilibrium')
ax.plot(v, f0 + f1,  lw=1.5, color='firebrick', ls='--', label=r'$f_0 + f_1$: perturbed')
ax.fill_between(v, f0, f0 + f1, alpha=0.25, color='firebrick', label=r'$f_1$ (small)')
ax.set_xlabel(r'$v / v_{th}$'); ax.set_ylabel(r'$f(v)$')
ax.set_title('Perturbation Ansatz'); ax.legend(fontsize=8); ax.grid(alpha=0.3)

# --- Panel 2: df0/dv — the driver of the coupling term ---
ax = axes[1]
df0dv = np.gradient(f0, v)
ax.plot(v, f0,    lw=2, color='steelblue', label=r'$f_0(v)$')
ax.plot(v, df0dv, lw=2, color='seagreen',  label=r'$\partial f_0/\partial v$', ls='--')
ax.axhline(0, color='k', lw=0.8)
ax.set_xlabel(r'$v / v_{th}$'); ax.set_ylabel(r'Amplitude')
ax.set_title(r'$\partial f_0/\partial v$: the coupling kernel')
ax.legend(fontsize=9); ax.grid(alpha=0.3)
ax.text(0.5, 0.05,
        r'This appears in $\hat{f}_1 \propto \frac{k\,\partial f_0/\partial v}{\omega - kv}$',
        transform=ax.transAxes, ha='center', fontsize=8.5,
        bbox=dict(boxstyle='round', fc='lightyellow', alpha=0.9))

# --- Panel 3: Resonance denominator 1/(v - vph) ---
ax = axes[2]
v_phi = 2.0
# Regularise near resonance for plotting
eps_reg = 0.08
denom = v - v_phi
resonance_weight = denom / (denom**2 + eps_reg**2)   # principal value
ax.plot(v, f0,              lw=2, color='steelblue', label=r'$f_0(v)$')
ax.plot(v, resonance_weight * 0.25, lw=2, color='darkorange',
        label=r'$\mathcal{P}\frac{1}{v-v_\phi}$ (PV)', ls='--')
ax.axvline(v_phi, color='firebrick', lw=2, ls=':', label=r'$v_\phi = \omega/k$')
ax.set_xlabel(r'$v / v_{th}$'); ax.set_ylabel(r'Amplitude')
ax.set_title('Resonance: Singular Denominator'); ax.legend(fontsize=8); ax.grid(alpha=0.3)

plt.suptitle(
    'Figure 5 — Linearisation: the guitar-string shiver of the plasma',
    fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

---
<a id='6'></a>
## 6. The Climax: Deriving Langmuir Waves and Landau Damping

### The Physical Narrative

When electrons are collectively displaced from their equilibrium position, the Coulomb attraction of the stationary ion background pulls them back. Their thermal inertia causes them to overshoot the equilibrium, generating a rhythmic, high-frequency oscillation — the **Langmuir wave**. It is the plasma's own heartbeat.

> **Analogy.** A pendulum released in a vacuum oscillates at a frequency determined by gravity (the restoring force) and its length (the inertia). A Langmuir wave oscillates at a frequency determined by the electrostatic restoring force (proportional to density $n_0$ through $\omega_{pe}$) and the electron inertia (mass $m_e$), with a thermal correction that stiffens the wave at shorter wavelengths.

### 6.1 The Landau Prescription: Resolving the Resonant Denominator

The integral in (5.8) is singular at $v = \omega/k$ (the resonant velocity). Landau (1946) resolved this ambiguity by recognising that the problem must be treated as an **initial-value problem**: the perturbation is switched on at $t = 0$, and the physical solution corresponds to taking $\mathrm{Im}(\omega) > 0$ in the Laplace transform (waves that were generated in the past, not artificial incoming waves from the future). When $\omega$ is analytically continued from the upper half-plane to the real axis, the **Plemelj–Sokhotski identity** gives:

$$\lim_{\gamma \to 0^+}\int_{-\infty}^{\infty}\frac{g(v)}{v - \omega_r/k - i\gamma/k}\,dv = \mathcal{P}\int_{-\infty}^{\infty}\frac{g(v)}{v - \omega_r/k}\,dv + i\pi\,g\!\left(\frac{\omega_r}{k}\right), \tag{6.1}$$

where $\mathcal{P}$ denotes the Cauchy principal value. Here $g(v) = \partial f_0/\partial v$.

### 6.2 The Bohm–Gross Dispersion Relation

For a Maxwellian equilibrium $f_0 = n_0/(\sqrt{2\pi}\,v_{Te})\,\exp(-v^2/2v_{Te}^2)$ and long-wavelength perturbations ($k\lambda_D \ll 1$), the principal-value integral can be evaluated by Taylor-expanding the denominator:

$$\mathcal{P}\int\frac{\partial f_0/\partial v}{v - v_\phi}\,dv \approx -\frac{n_0}{v_\phi^2}\left(1 + \frac{3v_{Te}^2}{v_\phi^2} + \ldots\right). \tag{6.2}$$

Substituting into $\mathrm{Re}[\varepsilon] = 0$ with $v_\phi = \omega_r/k$:

$$\boxed{\omega_r^2 = \omega_{pe}^2 + 3k^2v_{Te}^2 \equiv \omega_{pe}^2(1 + 3k^2\lambda_D^2).} \tag{6.3}$$

**Variable anatomy:**
- $\omega_r$ — real part of the wave frequency $[\mathrm{rad\,s}^{-1}]$. The oscillation rate.
- $\omega_{pe} = \sqrt{n_0 e^2/m_e\epsilon_0}$ — electron plasma frequency $[\mathrm{rad\,s}^{-1}]$. The fundamental oscillation frequency of the electron gas; depends only on density.
- $k$ — wave number $[\mathrm{m}^{-1}]$.
- $v_{Te} = \sqrt{T_e/m_e}$ — electron thermal velocity $[\mathrm{m\,s}^{-1}]$. Sets the thermal correction.
- $\lambda_D = v_{Te}/\omega_{pe}$ — Debye length $[\mathrm{m}]$.

**Interpretation.** The first term $\omega_{pe}^2$ represents the pure electrostatic restoring force; the term $3k^2v_{Te}^2$ is the thermal pressure correction — finite temperature makes the wave stiffer (higher frequency) at shorter wavelengths. In the cold limit $T_e \to 0$, all Langmuir waves oscillate at the single frequency $\omega_{pe}$.

### 6.3 Landau Damping

The imaginary part of the dispersion relation (5.8), via (6.1), gives the **damping rate** perturbatively ($|\gamma| \ll \omega_r$):

$$\gamma = -\frac{\pi\omega_{pe}^3}{2k^2}\frac{\partial f_0}{\partial v}\bigg|_{v = \omega_r/k}. \tag{6.4}$$

For a Maxwellian, $\partial f_0/\partial v|_{v_\phi} = -(v_\phi/v_{Te}^2)f_0(v_\phi) < 0$, so $\gamma < 0$: **the wave decays**.

Substituting the Maxwellian explicitly:

$$\boxed{\gamma = -\sqrt{\frac{\pi}{8}}\,\frac{\omega_{pe}}{(k\lambda_D)^3}\exp\!\left(-\frac{1}{2k^2\lambda_D^2} - \frac{3}{2}\right).} \tag{6.5}$$

**Physical mechanism — the resonant surfers.** Particles with velocity $v \approx v_\phi = \omega_r/k$ (the wave phase velocity) interact resonantly with the wave. Slightly slower particles are accelerated by the wave's electric field (they surf the wave); slightly faster particles are decelerated (they push back against the wave). For a Maxwellian, there are more slow than fast resonant particles (since $\partial f_0/\partial v < 0$ at $v_\phi$), so the net energy transfer is from **wave to particles**: the wave damps. Crucially, no collisions are involved — this is a collisionless, reversible (but exponentially fast) damping mechanism.

**Instability.** If $\partial f_0/\partial v|_{v_\phi} > 0$ (more fast than slow particles at the resonance, as in a beam distribution), then $\gamma > 0$: the wave grows exponentially. This is the **bump-on-tail instability**, the kinetic mechanism by which free energy in a non-Maxwellian distribution is converted into wave energy.

In [ ]:
# ── Figure 6: Langmuir wave dispersion and Landau damping (numerical + analytic)

def dielectric(klD, omega_norm):
    """Maxwellian electrostatic dielectric ε(k, ω).

    Parameters
    ----------
    klD        : float, k * lambda_D
    omega_norm : complex, ω / ω_pe

    Returns
    -------
    ε : complex
    """
    zeta = omega_norm / (np.sqrt(2) * klD)
    return 1.0 - (1.0 / (2.0 * klD**2)) * (1.0 + zeta * plasma_Z(zeta))


klD_arr = np.linspace(0.05, 0.60, 220)
omega_r_num, gamma_num = [], []

for klD in klD_arr:
    # Analytic initial guesses
    wr0 = np.sqrt(1.0 + 3.0 * klD**2)
    gi0 = -np.sqrt(np.pi / 8.0) / klD**3 * np.exp(-1.0 / (2.0 * klD**2) - 1.5)

    def equations(x):
        eps = dielectric(klD, x[0] + 1j * x[1])
        return [eps.real, eps.imag]

    sol = fsolve(equations, [wr0, gi0], full_output=False)
    omega_r_num.append(sol[0])
    gamma_num.append(sol[1])

omega_r_num = np.array(omega_r_num)
gamma_num   = np.array(gamma_num)

# Analytic approximations
omega_BG = np.sqrt(1.0 + 3.0 * klD_arr**2)   # Bohm–Gross
gamma_an = -np.sqrt(np.pi / 8.0) / klD_arr**3 * np.exp(-1.0 / (2.0 * klD_arr**2) - 1.5)

fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))

# --- Panel 1: Resonance picture ---
ax = axes[0]
v_plot = np.linspace(-5, 5, 500)
fM = maxwellian_1d(v_plot)
v_phi_mark = 2.1
ax.plot(v_plot, fM, 'k', lw=2.5, label=r'$f_0(v)$: Maxwellian')
ax.axvline(v_phi_mark, color='firebrick', lw=2, ls='--',
           label=r'$v_\phi = \omega_r/k$')
ax.fill_between(v_plot, fM,
                where=(v_plot > v_phi_mark - 0.55) & (v_plot < v_phi_mark),
                alpha=0.35, color='steelblue', label=r'Slow resonant: gain energy')
ax.fill_between(v_plot, fM,
                where=(v_plot > v_phi_mark) & (v_plot < v_phi_mark + 0.55),
                alpha=0.35, color='firebrick', label=r'Fast resonant: lose energy')
ax.text(0.03, 0.96,
        r'$\partial f_0/\partial v|_{v_\phi} < 0$' + '\n→ Landau damping',
        transform=ax.transAxes, fontsize=8.5, va='top',
        bbox=dict(boxstyle='round', fc='lightyellow', alpha=0.95))
ax.set_xlabel(r'$v / v_{th}$'); ax.set_ylabel(r'$f_0(v)$')
ax.set_title('Wave–Particle Resonance')
ax.legend(fontsize=7.5); ax.grid(alpha=0.3)

# --- Panel 2: Dispersion relation ---
ax = axes[1]
ax.plot(klD_arr, omega_r_num, 'b',   lw=2.5, label='Vlasov (numerical)')
ax.plot(klD_arr, omega_BG,    'b--', lw=1.8, label='Bohm–Gross (analytic)')
ax.axhline(1.0, color='gray', lw=1, ls=':')
ax.text(0.52, 1.01, r'$\omega_{pe}$', fontsize=10, color='gray')
ax.set_xlabel(r'$k\lambda_D$'); ax.set_ylabel(r'$\omega_r / \omega_{pe}$')
ax.set_title('Langmuir Wave Dispersion')
ax.set_xlim(0, 0.62); ax.set_ylim(0.9, 2.3)
ax.legend(); ax.grid(alpha=0.3)

# --- Panel 3: Landau damping rate ---
ax = axes[2]
ax.semilogy(klD_arr, np.abs(gamma_num), 'r',   lw=2.5, label=r'$|\gamma|$ (numerical)')
ax.semilogy(klD_arr, np.abs(gamma_an),  'r--', lw=1.8, label=r'$|\gamma|$ (analytic)')
ax.set_xlabel(r'$k\lambda_D$'); ax.set_ylabel(r'$|\gamma| / \omega_{pe}$')
ax.set_title('Landau Damping Rate')
ax.set_ylim(1e-7, 2); ax.legend(); ax.grid(alpha=0.3)

plt.suptitle(
    'Figure 6 — Langmuir Waves: dispersion relation and Landau damping',
    fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── Figure 7: Bump-on-tail instability — inverse Landau damping ───────────────
v_plot = np.linspace(-4, 6, 600)
f_core = maxwellian_1d(v_plot, vth=1.0)
f_beam = 0.10 * maxwellian_1d(v_plot, vth=0.45, vd=3.4)
f_tot  = f_core + f_beam
dfdv   = np.gradient(f_tot, v_plot)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

# --- Panel 1: Beam + core distribution ---
ax = axes[0]
ax.plot(v_plot, f_core, lw=2,   color='steelblue', ls='--', label='Core')
ax.plot(v_plot, f_beam, lw=2,   color='firebrick',  ls='--', label='Beam')
ax.plot(v_plot, f_tot,  lw=2.5, color='k',          label='Total $f$')
ax.fill_between(v_plot, f_tot, where=dfdv > 0, alpha=0.28, color='gold',
                label=r'$\partial f/\partial v > 0$: instability')
ax.set_xlabel(r'$v / v_{th}$'); ax.set_ylabel(r'$f(v)$')
ax.set_title('Bump-on-Tail: Unstable Distribution')
ax.legend(fontsize=9); ax.grid(alpha=0.3)

# --- Panel 2: Wave amplitude E(t) for stable vs unstable ---
ax = axes[1]
t = np.linspace(0, 9, 500)
E_stable  = np.exp(-0.28 * t) * np.cos(2.1 * t)   # Landau damped
E_growing = np.exp(0.22  * t) * np.cos(1.9 * t)   # Beam-driven growth
ax.plot(t, E_stable,  lw=2, color='steelblue', label=r'Maxwellian: $\gamma < 0$ (damped)')
ax.plot(t, E_growing, lw=2, color='firebrick',  label=r'Bump-on-tail: $\gamma > 0$ (growing)')
ax.axhline(0, color='k', lw=0.7)
ax.set_xlabel(r'$t\,\omega_{pe}$'); ax.set_ylabel(r'$E(t)$ [arb.]')
ax.set_title('Wave Amplitude: Damping vs. Growth')
ax.legend(fontsize=9); ax.grid(alpha=0.3)

plt.suptitle(
    'Figure 7 — Bump-on-Tail Instability: the kinetic free-energy mechanism',
    fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

---
<a id='7'></a>
## 7. Bridging to Code: The Need for Numerical Methods

### 7.1 The Limits of Linear Theory

The linearised analysis of Section 5 gives a complete picture of small-amplitude plasma waves: their dispersion, their damping rate, and their stability threshold. It fails, however, as soon as the perturbation amplitude grows large enough that the discarded second-order term $f_1 E_1$ becomes comparable to the retained terms. Specifically, linear theory breaks down when

$$|f_1| \sim f_0 \quad \text{or equivalently} \quad e|\phi_1|/T_e \sim 1. \tag{7.1}$$

Beyond this threshold, one must contend with nonlinear phenomena: particle trapping in wave troughs, mode coupling, wave-breaking, and the nonlinear saturation of kinetic instabilities. These require direct numerical solution of the full Vlasov–Poisson system.

### 7.2 Choice of Numerical Method

The central challenge is representing and evolving the distribution function $f(x, v, t)$ on a 2D (for 1D-1V) or higher-dimensional grid.

**Particle-in-Cell (PIC)** methods represent $f$ by a finite number of macro-particles. They are computationally efficient but introduce **statistical noise** of order $1/\sqrt{N_p}$, which can mask the exponentially small Landau damping signal, especially at long wavelengths where $|\gamma|/\omega_{pe} \sim e^{-1/(2k^2\lambda_D^2)} \ll 1$.

**Semi-Lagrangian (Eulerian grid)** methods discretise $f$ on a fixed $(x, v)$ mesh and use the characteristic equation $df/dt|_{\rm orbit} = 0$ to advect $f$ without statistical noise. The Vlasov equation becomes

$$f^{n+1}(x_i, v_j) = f^n\!\left(x_i - v_j\Delta t,\; v_j - \frac{qE_i^n}{m}\Delta t\right), \tag{7.2}$$

advecting the distribution backwards along characteristics and interpolating. This approach achieves spectral-level accuracy in velocity space and is the preferred method for precision Langmuir wave studies.

### 7.3 Conservation as a Sanity Check

A well-posed numerical scheme must respect the global conservation laws of the Vlasov–Poisson system:

$$N_e = \int\!\int f\,dx\,dv = \mathrm{const.}, \tag{7.3}$$

$$W = \underbrace{\frac{m_e}{2}\int\!\int v^2 f\,dx\,dv}_{\text{kinetic}} + \underbrace{\frac{\epsilon_0}{2}\int E^2\,dx}_{\text{field}} = \mathrm{const.} \tag{7.4}$$

These identities serve as **necessary consistency checks** of any simulation: a code that violates particle or energy conservation — even at the percent level — cannot be trusted to reproduce Landau damping, whose signal is often orders of magnitude smaller than the total energy.

The Casimir invariants (Section 3.5), particularly $\int\!\int f^2\,dx\,dv = \mathrm{const.}$, provide additional high-sensitivity checks: any numerical diffusion or interpolation error that thermalises $f$ will violate the $L^2$ norm before it detectably changes the energy.

### 7.4 Summary of the Theory–Simulation Connection

| Theoretical result | Role in simulation |
|--------------------|-------------------|
| Vlasov–Poisson system | Governing equations to be discretised |
| $df/dt|_{\rm orbit} = 0$ | Basis of the semi-Lagrangian advection step |
| Bohm–Gross: $\omega_r(k)$ | Target dispersion relation to be recovered from simulation |
| Landau rate: $\gamma(k)$ | Target decay rate; tests resolution of velocity grid |
| Conservation laws (7.3)–(7.4) | Quantitative sanity checks at every timestep |
| Casimir invariant $\int f^2$ | Sensitive check of numerical diffusion |

In [ ]:
# ── Figure 8: Summary — full theoretical and numerical pipeline ────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Panel 1: Conservation law schematic ---
ax = axes[0]
t_sim = np.linspace(0, 50, 300)
# Schematic "good" vs "leaky" integrator
E_total_good  = 1.0 + 0.005 * np.sin(0.3 * t_sim) * np.exp(-0.01 * t_sim)
E_total_bad   = 1.0 - 0.002 * t_sim + 0.05 * np.random.default_rng(7).standard_normal(300)
ax.plot(t_sim, E_total_good, 'steelblue', lw=2, label='Good integrator (conserved)')
ax.plot(t_sim, E_total_bad,  'firebrick',  lw=2, ls='--', label='Leaky integrator (drifts)')
ax.axhline(1.0, color='k', lw=1, ls=':')
ax.set_xlabel(r'$t\,\omega_{pe}$')
ax.set_ylabel(r'$W(t) / W(0)$')
ax.set_title('Total Energy Conservation: Sanity Check')
ax.legend(); ax.grid(alpha=0.3)
ax.set_ylim(0.9, 1.1)

# --- Panel 2: Summary table ---
ax = axes[1]
ax.axis('off')
rows = [
    ['Section', 'Key Result', 'Role'],
    ['0', r'$\Lambda = n\lambda_D^3 \gg 1$', 'Vlasov validity criterion'],
    ['1', r'$dN = f\,d^3r\,d^3v$',          'Phase-space density'],
    ['2', 'Klimontovich → Vlasov',           'Mean-field reduction'],
    ['3', r'$df/dt|_{\rm orbit}=0$',         'Liouville / incompressibility'],
    ['4', r'$\nabla^2\phi = -\rho/\epsilon_0$', 'Self-consistent fields'],
    ['5', r'$\varepsilon(k,\omega) = 0$',    'Dispersion relation'],
    ['6', r'$\omega^2 = \omega_{pe}^2+3k^2v_{Te}^2$', 'Langmuir wave + Landau damping'],
    ['7', 'Conservation checks',             'Simulation validation'],
]
tbl = ax.table(cellText=rows[1:], colLabels=rows[0],
               cellLoc='center', loc='center', bbox=[0, 0, 1, 1])
tbl.auto_set_font_size(False)
tbl.set_fontsize(9)
for (r, c_), cell in tbl.get_celld().items():
    if r == 0:
        cell.set_facecolor('#1d4ed8')
        cell.set_text_props(color='white', fontweight='bold')
    elif r % 2 == 0:
        cell.set_facecolor('#dbeafe')
    cell.set_edgecolor('#93c5fd')
ax.set_title('Theory Notebook: Section Summary', pad=12, fontweight='bold')

plt.suptitle('Figure 8 — Bridging Theory to Simulation',
             fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print("\n" + "=" * 65)
print("Theory Notebook complete.")
print("=" * 65)